# mini-vLLM — CARL Evaluation Runner (Colab)

Heavy CARL evaluation cells split out of `run_benchmarks.ipynb` so each notebook finishes within Colab's runtime limits. Run `run_benchmarks.ipynb` for the fast kernel benchmarks; run this one for the real-inference CARL evaluation suite.

In [ ]:
# 1. Setup: install deps, clone/sync mini-vLLM, put it on the path.
!pip install -q transformers accelerate datasets tqdm
!if [ ! -d "mini-vllm" ]; then git clone https://github.com/vsvidhun06-blip/mini-vllm.git; fi
%cd mini-vllm
!git pull origin main
import sys; sys.path.insert(0, '/content/mini-vllm')

# Helper + execution context shared by the CARL cells below. The run() helper is
# copied verbatim from run_benchmarks.ipynb; REPO_ROOT/env are the working dir +
# PYTHONPATH it needs, and results{} collects each benchmark's captured output,
# so this notebook is self-contained after the split.
import os, subprocess
REPO_ROOT = os.path.abspath("/content/mini-vllm")
env = {**os.environ, "PYTHONPATH": REPO_ROOT}
results = {}

def run(cmd):
    p = subprocess.run(cmd, cwd=REPO_ROOT, env=env, capture_output=True, text=True)
    print("$", " ".join(cmd))
    print(p.stdout)
    if p.returncode != 0:
        print("[exit %d] stderr:\n%s" % (p.returncode, p.stderr[-4000:]))
    return {"cmd": " ".join(cmd), "returncode": p.returncode,
            "stdout": p.stdout, "stderr": p.stderr}

print("Setup complete")

In [ ]:
# 6c. CARL — REAL TinyLlama inference, adaptive controller vs fixed baseline.
#     Unlike scripts/benchmark_carl.py's default mode (a torch-free control-loop
#     SIMULATION over an analytical cost model), `--live` runs the actual serving
#     engine: TinyLlama weights, the real ContinuousBatchScheduler, real
#     prefill/decode forward passes. CARL drives the scheduler's max_batch_size /
#     chunk_size LIVE while requests are served, with speculative decoding pinned
#     OFF (TinyLlama self-spec is below break-even, so it would only add noise).
#
#     Three scenarios x 50 requests, each served twice (baseline vs carl_adaptive):
#       INTERACTIVE     short prompts (16-32 tok),  max_new=32
#       BATCH           long prompts  (128-256 tok), max_new=64
#       NON-STATIONARY  25 INTERACTIVE then 25 BATCH injected mid-run (no notice)
#     Reports throughput (tok/s) and P50/P99 TTFT & TPOT, and writes
#     docs/carl_live_results.json (a SEPARATE file from the simulation's
#     docs/carl_results.json, so neither artifact clobbers the other).
#     This one needs the GPU — it's real generation, so run it after the model
#     is cached (cell 4). Folded into `results` so it lands in docs/benchmarks.md.
#
#     COLAB / fp16 NOTE: on a T4 the model serves in fp16, but the from-scratch
#     Triton FA2 kernel (src/engine/kernels/flash_attention.py) runs ENTIRELY in
#     fp32 internally — flash_attention_forward() promotes fp16 q/k/v (and any
#     additive mask) to fp32 on entry and casts the output back to fp16 on exit.
#     Triton's tl.dot returns its result in the input operands' dtype, so feeding
#     it fp16 made the scores fp16 and the value matmul then mixed fp16/fp32
#     ("Both operands must be same dtype" crash); promoting up front sidesteps it
#     and keeps numerics on the engine's true-fp32 (TF32-off) parity path. If you
#     still hit that error here, you're running STALE code — re-run cell 3, which
#     now hard-syncs to origin and purges cached .pyc files.
_carl_key = "CARL — real TinyLlama: adaptive vs baseline"
_carl = run(["python", "scripts/benchmark_carl.py", "--live", "--limit", "50"])
results[_carl_key] = _carl

# Verbose failure surfacing. This is the most fragile cell — it drives real GPU
# kernels — and run() only prints the last 4000 chars of stderr; for a Triton
# compile error that tail is often just generated IR, hiding the real cause. So
# if the live harness produced no metrics table (no "tok/s" line) OR exited
# non-zero, dump the FULL stdout and stderr here so the failure is visible.
_carl_ok = _carl["returncode"] == 0 and "tok/s" in _carl["stdout"]
if not _carl_ok:
    print("\n" + "!" * 72)
    print("CARL live benchmark produced NO results table — full diagnostics below")
    print("  exit code            :", _carl["returncode"])
    print("  'tok/s' seen in stdout:", "tok/s" in _carl["stdout"])
    print("!" * 72)
    print("\n----- FULL STDOUT -----\n" + (_carl["stdout"] or "(empty)"))
    print("\n----- FULL STDERR -----\n" + (_carl["stderr"] or "(empty)"))
    print("----- END DIAGNOSTICS -----")
else:
    print("CARL live benchmark OK — metrics table captured for docs/benchmarks.md.")


In [ ]:
# 6e. CARL ablation on REAL TinyLlama inference (the MEASURED counterpart to 6d).
#     6d runs the torch-free SIMULATION ablation; this runs ten ablation/baseline
#     configs (CARL-Full, the five CARL-NoX, Static-Best, AutoTuner, CARL-Thompson,
#     DynOracle) through the actual serving engine on the GPU, over NON-STATIONARY
#     (INTERACTIVE -> BATCH), and reports measured throughput / TTFT / TPOT
#     (mean +/- std). Static-Best is chosen by a held-out Latin-Hypercube
#     validation search (seed 999); DynOracle is the retrospective offline
#     upper bound from CARL-Full's recorded rewards.
#
#     HONEST LIMITATION: in this single-model live harness CARL is wired to the
#     SCHEDULER ONLY, speculation is pinned off, there is no router, and KV
#     eviction never triggers at these sizes -- so only CARL-NoSched / CARL-NoChunk
#     (and Static-Best / AutoTuner / DynOracle) differ from CARL-Full; NoSpec /
#     NoCache / NoRouter are EXPECTED to match CARL-Full (the table marks them
#     'no*' in the 'live?' column). That is the honest hardware result -- it shows
#     the SCHEDULER is what moves real single-GPU TinyLlama metrics; the simulation
#     ablation (6d) is the one that can vary all five subsystems. See
#     docs/eval/README.md.
#
#     This is the heaviest cell, so run it after the model is cached (cell 4).
#     PREVIEW here uses N=3 seeds x 30 requests; the paper table is produced by
#     the default N=10 seeds (42..51) x 50 requests. Writes
#     docs/eval/ablation_live_results.json and folds into docs/benchmarks.md.
results["CARL ablation — real TinyLlama (live)"] = run(
    ["python", "scripts/eval/ablation_live.py", "--seeds", "42,43,44", "--limit", "30"]
)

# Render the live-ablation figure from the measured JSON.
run(["python", "scripts/plot/generate_ablation_figure.py"])


In [ ]:
# 6f. Cross-model CARL validation: does the SAME controller + SAME
#     DEFAULT_CONFIGS generalize across model families with ZERO retuning?
#     Loads each model through the from-scratch engine (now LLaMA + Qwen2
#     aware) and serves the identical NON-STATIONARY workload under CARL-Full,
#     Static-Best (held-out LHS validation), and AutoTuner; N=3 seeds x 50 req.
#
#     INVARIANT (stated in code + output): "Same CARLConfig DEFAULT_CONFIGS
#     used for all models without modification." The primary metric is
#     normalized_performance = CARL-Full throughput / Static-Best throughput,
#     aggregated per model (mean/std/CI95) and then averaged ACROSS MODELS.
#
#     Models the from-scratch engine cannot represent exactly (Gemma's GeGLU
#     + embedding scaling) or that don't fit in VRAM are skipped gracefully
#     (loaded=false + skip_reason) and excluded from the headline. TinyLlama
#     and Qwen2-0.5B load natively; gemma-2b-it is attempted and skipped if it
#     can't load. No oracle gap is computed here (that lives in the ablation).
#
#     Heaviest cell: loads up to three models sequentially. Writes
#     docs/eval/cross_model_results.json (+ raw under docs/eval/raw/cross_model/).
results["CARL cross-model validation"] = run(
    ["python", "scripts/eval/cross_model.py", "--seeds", "42,43,44", "--limit", "50"]
)


In [ ]:
# 6g. CARL honest failure analysis: where does the adaptive controller LOSE,
#     and is the stated reason the REAL reason? This is the candid counterpart
#     to the wins reported in 6e/6f -- a strong-baseline (Static-Best) study
#     that surfaces scenarios where CARL-Full ties or underperforms, logs each
#     failure with a pre-registered hypothesis (failure_reason_hypothesis) and
#     the post-hoc verdict after inspecting the adaptation trace
#     (failure_reason_final: CONFIRMED or REVISED), plus the latency tradeoff
#     and the controller's arm_changes / unique_arms churn.
#
#     N=3 seeds x 30 requests here for the Colab preview; the paper figure uses
#     the default N=10 seeds (42..51) x 50 requests. Writes
#     docs/eval/failure_cases_results.json (+ raw under
#     docs/eval/raw/failure_cases/) and folds into docs/benchmarks.md.
results["CARL failure cases"] = run(
    ["python", "scripts/eval/failure_cases.py",
     "--seeds", "42,43,44", "--limit", "30"]
)


In [ ]:
# 6. Download results: zip every eval artifact under docs/eval and pull it down.
import shutil, os
from google.colab import files
# Zip all eval results
shutil.make_archive('/content/carl_eval_results', 'zip',
                    '/content/mini-vllm/docs/eval')
files.download('/content/carl_eval_results.zip')
print("Downloaded carl_eval_results.zip")